In [0]:
import os
import sys
import yaml
import argparse
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
from pyspark.sql import SparkSession

# 1. Parse DABs injected parameters (with Notebook fallback)
parser = argparse.ArgumentParser()
parser.add_argument("--env", type=str, default="dev", help="Deployment environment (dev/prod)")
parser.add_argument("--catalog", type=str, default="maven_market_uc", help="Target Unity Catalog")

if hasattr(sys, 'ps1') or 'IPython' in sys.modules:
    print("[INFO] Interactive Notebook environment detected. Using default dev arguments.")
    args, _ = parser.parse_known_args(args=["--env", "dev", "--catalog", "maven_market_uc"])
else:
    args, _ = parser.parse_known_args()

# 2. Safely resolve script directory and locate config.yml
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

REPO_ROOT = SCRIPT_DIR
while REPO_ROOT != os.path.dirname(REPO_ROOT):
    if os.path.exists(os.path.join(REPO_ROOT, "config.yml")):
        break
    REPO_ROOT = os.path.dirname(REPO_ROOT)

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from utils.custom_logger import CustomLogger

# 3. Initialize Spark Session & Logger
spark = SparkSession.builder.getOrCreate()
config_path = os.path.join(REPO_ROOT, "config.yml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

audit_table = config.get("tables", {}).get("audit_logs_table", f"{args.catalog}.audit.pipeline_audit_logs")
logger = CustomLogger(spark, f"kafka_to_uc_bronze_{args.env}", target_table=audit_table)
logger.log_start()

try:
    # 4. Load Kafka Configuration
    kafka_config = config["sources"]["kafka"]
    bootstrap_servers = kafka_config["bootstrap_servers"]
    topic_orders = next(s["topic"] for s in kafka_config["streams"] if "orders" in s["topic"])
    topic_inventory = next(s["topic"] for s in kafka_config["streams"] if "inventory" in s["topic"])
    
    kafka_key = dbutils.secrets.get(scope=kafka_config["secret_scope"], key=kafka_config["api_key_name"])
    kafka_secret = dbutils.secrets.get(scope=kafka_config["secret_scope"], key=kafka_config["api_secret_name"])

    jaas_config = (
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{kafka_key}" password="{kafka_secret}";'
    )

    # 5. Define Schemas to match Synthetic Generator exact types
    orders_schema = StructType([
        StructField("order_id", StringType(), True),
        StructField("transaction_date", StringType(), True),
        StructField("stock_date", StringType(), True),
        StructField("product_id", IntegerType(), True),
        StructField("customer_id", IntegerType(), True),
        StructField("store_id", IntegerType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("unit_price", DoubleType(), True),
        StructField("event_timestamp", StringType(), True)
    ])

    inventory_schema = StructType([
        StructField("event_id", StringType(), True),
        StructField("store_id", IntegerType(), True),
        StructField("product_id", IntegerType(), True),
        StructField("stock_level", IntegerType(), True),
        StructField("reorder_threshold", IntegerType(), True),
        StructField("restock_flag", BooleanType(), True),
        StructField("event_timestamp", StringType(), True)
    ])

    # 6. Reusable Function to Read from Kafka
    def read_from_kafka(topic_name):
        return (spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", bootstrap_servers)
            .option("subscribe", topic_name)
            .option("startingOffsets", "earliest")
            .option("kafka.security.protocol", "SASL_SSL")
            .option("kafka.sasl.mechanism", "PLAIN")
            .option("kafka.sasl.jaas.config", jaas_config)
            .load()
        )

    # 7. Ingest, Parse, and add required Bronze Metadata
    parsed_orders_df = read_from_kafka(topic_orders) \
        .select(F.from_json(F.col("value").cast(StringType()), orders_schema).alias("data")) \
        .select("data.*") \
        .withColumn("_ingested_at", F.current_timestamp()) \
        .withColumn("_source", F.lit(f"kafka_{topic_orders}"))

    parsed_inventory_df = read_from_kafka(topic_inventory) \
        .select(F.from_json(F.col("value").cast(StringType()), inventory_schema).alias("data")) \
        .select("data.*") \
        .withColumn("_ingested_at", F.current_timestamp()) \
        .withColumn("_source", F.lit(f"kafka_{topic_inventory}"))

    # 8. Setup Unity Catalog Volume using dynamic catalog
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {args.catalog}.bronze.checkpoints")
    
    # 9. Map target tables from config safely using .get() fallbacks
    base_catalog = config.get("catalog", "maven_market_uc")
    
    target_table_orders = config.get("tables", {}).get(
        "bronze_kafka_orders", f"{base_catalog}.bronze.raw_kafka_orders"
    ).replace(base_catalog, args.catalog)
    
    target_table_inventory = config.get("tables", {}).get(
        "bronze_kafka_inventory", f"{base_catalog}.bronze.raw_kafka_inventory"
    ).replace(base_catalog, args.catalog)

    checkpoint_orders = config["sources"]["kafka"]["streams"][0]["checkpoint_location"]
    checkpoint_inventory = config["sources"]["kafka"]["streams"][1]["checkpoint_location"]

    orders_query = (
        parsed_orders_df
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_orders)
        .trigger(availableNow=True)
        .toTable(target_table_orders)
    )

    inventory_query = (
        parsed_inventory_df
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_inventory)
        .trigger(availableNow=True)
        .toTable(target_table_inventory)
    )

    print("Streaming initialized. Processing available data in Kafka and writing to Bronze tables...")
    
    # Block until ALL streams finish
    for stream in spark.streams.active:
        stream.awaitTermination()
    
    rec_count_orders = spark.table(target_table_orders).count()
    rec_count_inventory = spark.table(target_table_inventory).count()
    total_records = rec_count_orders + rec_count_inventory
    
    logger.log_success(records_processed=total_records)
    print("All available data has been processed and streams have successfully terminated.")

except Exception as e:
    logger.log_failure(e)
    raise e